In [7]:
import pandas as pd
import re

# ========= 1. 读取原始数据 =========
# 把这里改成你的文件名
df = pd.read_csv("metadataset.csv")

print("原始数据条数:", len(df))
print("列名:", df.columns.tolist())

# ========= 2. 基础清洗 =========
# 只保留 Text 类型 + 英文
base = df[
    (df["Type"].fillna("").str.lower() == "text") &
    (df["Language"].fillna("").str.contains(r"\ben\b", case=False, na=False))
].copy()

print("基础筛选后条数:", len(base))

# ========= 3. 定义 Yes 关键词 =========
# 这些是更符合 meaningful reading material 的关键词
yes_keywords = [
    "children",
    "children's stories",
    "juvenile",
    "fairy tales",
    "fiction",
    "literature",
    "stories",
    "readers",
    "education",
    "school",
    "short stories",
    "novel",
    "poetry",
    "drama",
    "essays",
    "literary collections"
]

yes_pattern = "|".join(re.escape(k) for k in yes_keywords)

# ========= 4. 定义排除关键词 =========
# 这些更像参考资料、索引、目录，不太适合作为 reading material
exclude_keywords = [
    "bibliography",
    "catalog",
    "catalogue",
    "dictionary",
    "index",
    "gazetteer",
    "periodical",
    "yearbook",
    "almanac",
    "encyclopedia",
    "encyclopaedia",
    "manual",
    "handbook",
    "law",
    "legislation",
    "statutes",
    "court",
    "regulations",
    "directory"
]

exclude_pattern = "|".join(re.escape(k) for k in exclude_keywords)

# ========= 5. 在多个字段里筛 Yes =========
# 优先看 Subjects / Bookshelves / Title
yes_candidates = base[
    base["Subjects"].fillna("").str.contains(yes_pattern, case=False, na=False) |
    base["Bookshelves"].fillna("").str.contains(yes_pattern, case=False, na=False) |
    base["Title"].fillna("").str.contains(yes_pattern, case=False, na=False)
].copy()

print("Yes 候选条数（未排除）:", len(yes_candidates))

# ========= 6. 排除明显不适合的内容 =========
yes_final = yes_candidates[
    ~yes_candidates["Subjects"].fillna("").str.contains(exclude_pattern, case=False, na=False) &
    ~yes_candidates["Bookshelves"].fillna("").str.contains(exclude_pattern, case=False, na=False) &
    ~yes_candidates["Title"].fillna("").str.contains(exclude_pattern, case=False, na=False)
].copy()

print("最终 Yes 条数:", len(yes_final))

# ========= 7. 去重 =========
yes_final = yes_final.drop_duplicates(subset=["Text#"])

print("去重后 Yes 条数:", len(yes_final))

# ========= 8. 导出结果 =========
# 完整版
yes_final.to_csv("gutenberg_yes_candidates_full.csv", index=False)

# 精简版，方便人工检查
yes_final[["Text#", "Title", "Authors", "Language", "Subjects", "Bookshelves"]].to_csv(
    "gutenberg_yes_candidates_small.csv", index=False
)

print("已导出:")
print("- gutenberg_yes_candidates_full.csv")
print("- gutenberg_yes_candidates_small.csv")

# ========= 9. 看看前 20 条 =========
print("\n前 20 条结果:")
print(
    yes_final[["Text#", "Title", "Authors", "Subjects", "Bookshelves"]]
    .head(20)
    .to_string(index=False)
)

原始数据条数: 78243
列名: ['Text#', 'Type', 'Issued', 'Title', 'Language', 'Authors', 'Subjects', 'LoCC', 'Bookshelves']
基础筛选后条数: 61015
Yes 候选条数（未排除）: 40223
最终 Yes 条数: 36242
去重后 Yes 条数: 36242
已导出:
- gutenberg_yes_candidates_full.csv
- gutenberg_yes_candidates_small.csv

前 20 条结果:
 Text#                                                                                                         Title                                                                  Authors                                                                                                                                                                                                                                                               Subjects                                                                                                                                                                         Bookshelves
     3                                                                           John F. Kenne

In [9]:
import os
import re
import time
import pandas as pd
import requests
from bs4 import BeautifulSoup

# ========= 1. 配置 =========
CSV_PATH = "gutenberg_yes_candidates_small.csv"
OUTPUT_DIR = "gutenberg_yes_texts"
MAX_DOWNLOADS = 30        # 先少量试跑，确认没问题后再改大
SLEEP_SECONDS = 1.5       # 每本书之间停一下，别太快

os.makedirs(OUTPUT_DIR, exist_ok=True)

headers = {
    "User-Agent": "Mozilla/5.0 (compatible; FIT5120-project/1.0)"
}

# ========= 2. 读取候选书单 =========
df = pd.read_csv(CSV_PATH)

# 确保有 Text# 列
df = df[df["Text#"].notna()].copy()
df["Text#"] = df["Text#"].astype(int)

# 去重
df = df.drop_duplicates(subset=["Text#"])

# 先只取前 MAX_DOWNLOADS 本
df = df.head(MAX_DOWNLOADS)

print(f"准备下载 {len(df)} 本书")

# ========= 3. 文件名清洗函数 =========
def safe_filename(text):
    text = str(text)
    text = re.sub(r'[\\/*?:"<>|]', "_", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text[:120]

# ========= 4. 找 txt 下载链接 =========
def find_txt_link(book_id):
    url = f"https://www.gutenberg.org/ebooks/{book_id}"
    r = requests.get(url, headers=headers, timeout=30)
    r.raise_for_status()

    soup = BeautifulSoup(r.text, "html.parser")

    # 优先找 Plain Text UTF-8
    for a in soup.find_all("a", href=True):
        text = a.get_text(" ", strip=True).lower()
        href = a["href"]

        if "plain text utf-8" in text:
            if href.startswith("/"):
                return "https://www.gutenberg.org" + href
            elif href.startswith("http"):
                return href

    # 其次找所有 txt 链接
    for a in soup.find_all("a", href=True):
        href = a["href"]
        if ".txt" in href:
            if href.startswith("/"):
                return "https://www.gutenberg.org" + href
            elif href.startswith("http"):
                return href

    return None

# ========= 5. 下载 txt =========
def download_txt(url, save_path):
    r = requests.get(url, headers=headers, timeout=60)
    r.raise_for_status()

    if r.encoding is None:
        r.encoding = "utf-8"

    with open(save_path, "w", encoding="utf-8", errors="ignore") as f:
        f.write(r.text)

# ========= 6. 批量下载 =========
success = []
failed = []

for _, row in df.iterrows():
    book_id = int(row["Text#"])
    title = safe_filename(row.get("Title", f"book_{book_id}"))
    save_path = os.path.join(OUTPUT_DIR, f"{book_id}_{title}.txt")

    if os.path.exists(save_path):
        print(f"[跳过] 已存在: {save_path}")
        success.append(book_id)
        continue

    try:
        print(f"[处理中] {book_id} - {title}")

        txt_link = find_txt_link(book_id)

        if not txt_link:
            print("  -> 没找到 txt 下载链接")
            failed.append((book_id, "No txt link found"))
            continue

        print(f"  -> 下载链接: {txt_link}")
        download_txt(txt_link, save_path)

        print("  -> 下载成功")
        success.append(book_id)

        time.sleep(SLEEP_SECONDS)

    except Exception as e:
        print(f"  -> 下载失败: {e}")
        failed.append((book_id, str(e)))

# ========= 7. 导出日志 =========
pd.DataFrame({"success_book_id": success}).to_csv("download_success.csv", index=False)
pd.DataFrame(failed, columns=["book_id", "error"]).to_csv("download_failed.csv", index=False)

print("\n下载完成")
print(f"成功: {len(success)}")
print(f"失败: {len(failed)}")
print(f"下载文件夹: {OUTPUT_DIR}")

准备下载 30 本书
[处理中] 3 - John F. Kennedy's Inaugural Address
  -> 下载链接: https://www.gutenberg.org/ebooks/3.txt.utf-8
  -> 下载成功
[处理中] 4 - Lincoln's Gettysburg Address Given November 19, 1863 on the battlefield near Gettysburg, Pennsylvania, USA
  -> 下载链接: https://www.gutenberg.org/ebooks/4.txt.utf-8
  -> 下载成功
[处理中] 6 - Give Me Liberty or Give Me Death
  -> 下载链接: https://www.gutenberg.org/ebooks/6.txt.utf-8
  -> 下载成功
[处理中] 8 - Abraham Lincoln's Second Inaugural Address
  -> 下载链接: https://www.gutenberg.org/ebooks/8.txt.utf-8
  -> 下载成功
[处理中] 9 - Abraham Lincoln's First Inaugural Address
  -> 下载链接: https://www.gutenberg.org/ebooks/9.txt.utf-8
  -> 下载成功
[处理中] 10 - The King James Version of the Bible
  -> 下载链接: https://www.gutenberg.org/ebooks/10.txt.utf-8
  -> 下载成功
[处理中] 11 - Alice's Adventures in Wonderland
  -> 下载链接: https://www.gutenberg.org/ebooks/11.txt.utf-8
  -> 下载成功
[处理中] 12 - Through the Looking-Glass
  -> 下载链接: https://www.gutenberg.org/ebooks/12.txt.utf-8
  -> 下载成功
[处理中] 13 - The Hunt

In [11]:
import shutil

# 把整个文件夹压缩成 zip
shutil.make_archive("gutenberg_yes_texts_bundle", "zip", "gutenberg_yes_texts")

print("压缩完成：gutenberg_yes_texts_bundle.zip")

压缩完成：gutenberg_yes_texts_bundle.zip
